# LogisticRegression on breast-bin.csv
`breast-bin.csv` has no header row, so it is read with `read_data_file` (numpy.genfromtxt).

In [1]:
import sys
from pathlib import Path

import numpy as np

# find the repository root (the folder that contains "datasets"), wherever the notebook is
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'datasets').is_dir())
for path in (ROOT / 'src', ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

DATA = ROOT / 'datasets'

from si.io.data_file import read_data_file
from si.model_selection.split import stratified_train_test_split
from si.models.logistic_regression import LogisticRegression

In [2]:
breast = read_data_file(str(DATA / 'breast_bin' / 'breast-bin.csv'), sep=',', label=True)
print(breast.shape(), np.unique(breast.y, return_counts=True))
train, test = stratified_train_test_split(breast, test_size=0.2, random_state=42)

(699, 9) (array([0., 1.]), array([458, 241]))


Which score (accuracy) and cost do you get?

In [3]:
model = LogisticRegression(l2_penalty=1.0, alpha=0.001, max_iter=2000, patience=5, scale=True)
model.fit(train)
print('iterations:', len(model.cost_history))
print('accuracy test:', round(model.score(test), 4))
print('cost test:', round(model.cost(test), 4))

iterations: 2000
accuracy test: 0.9496
cost test: 0.1996


## Baseline
Always predicting the majority class.

In [4]:
majority = np.bincount(train.y.astype(int)).argmax()
print('baseline accuracy:', round(np.mean(test.y == majority), 4))

baseline accuracy: 0.6547


## Learning rate

In [5]:
for alpha in [0.0001, 0.001, 0.01, 0.1]:
    m = LogisticRegression(alpha=alpha, max_iter=2000).fit(train)
    print(f'alpha={alpha:<7} iterations={len(m.cost_history):>5}  accuracy={m.score(test):.4f}  cost={m.cost(test):.4f}')

alpha=0.0001  iterations= 2000  accuracy=0.9496  cost=0.5272
alpha=0.001   iterations= 2000  accuracy=0.9496  cost=0.1996


alpha=0.01    iterations= 2000  accuracy=0.9568  cost=0.0985


alpha=0.1     iterations= 2000  accuracy=0.9784  cost=0.0824
